Multi-Personality Debate System using ChatPrompt

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [3]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [5]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.0)
from langchain_core.prompts import ChatPromptTemplate

In [11]:
SYS_PROMPT = """
        You are a debate moderator.

        Conduct a discussion between experts.

        Highlight:
        - Agreements
        - Disagreements
        - Counter arguments

        Generate a realistic debate transcript in 50 words.
        """

    
HUMAN_PROMPT =  """
                Question:
                {question}

                Expert Opinions:
                {opinions}
                """
      

In [14]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        ("human", HUMAN_PROMPT),
    ]
)
prompt

ChatPromptTemplate(input_variables=['opinions', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n        You are a debate moderator.\n\n        Conduct a discussion between experts.\n\n        Highlight:\n        - Agreements\n        - Disagreements\n        - Counter arguments\n\n        Generate a realistic debate transcript in 50 words.\n        '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['opinions', 'question'], input_types={}, partial_variables={}, template='\n                Question:\n                {question}\n\n                Expert Opinions:\n                {opinions}\n                '), additional_kwargs={})])

In [13]:
chain = (prompt
            |
           llm )

In [15]:
debate_res = chain.invoke({"question":"Should AI replace software developers?","opinions":"Software Engineer"})
print(debate_res)

content="**Moderator:** Welcome, experts. Should AI replace software developers?\n\n**Software Engineer:** AI can enhance productivity but shouldn't replace developers. Human creativity is irreplaceable.\n\n**AI Specialist:** I disagree. AI can automate repetitive tasks, allowing developers to focus on innovation.\n\n**Software Engineer:** True, but over-reliance on AI may stifle creativity and critical thinking.\n\n**AI Specialist:** Balance is key; AI should complement, not replace." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 76, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_85fd54540d', 'finish_reason': 'stop', 'logprobs': None} id='run--55a0c4a5-8a00-41e2-a853-38a005b6a0

In [18]:
from IPython.display import display, Markdown
display(Markdown(debate_res.content))

**Moderator:** Welcome, experts. Should AI replace software developers?

**Software Engineer:** AI can enhance productivity but shouldn't replace developers. Human creativity is irreplaceable.

**AI Specialist:** I disagree. AI can automate repetitive tasks, allowing developers to focus on innovation.

**Software Engineer:** True, but over-reliance on AI may stifle creativity and critical thinking.

**AI Specialist:** Balance is key; AI should complement, not replace.

In [19]:
debate_res = chain.invoke({"question":"Should AI replace software developers?","opinions":"CEO"})
display(Markdown(debate_res.content))

**Moderator:** Welcome, everyone. Should AI replace software developers? 

**CEO:** AI can enhance productivity but shouldn't fully replace developers. 

**Developer:** I agree; AI is a tool, not a replacement. 

**Tech Analyst:** Disagree. AI can automate coding tasks, reducing the need for human developers. 

**CEO:** But creativity and problem-solving are uniquely human traits. 

**Developer:** Exactly, AI lacks the intuition needed for complex projects. 

**Moderator:** Thank you!

In [20]:
debate_res = chain.invoke({"question":"Should AI replace software developers?","opinions":"Economist"})
display(Markdown(debate_res.content))

**Moderator:** Welcome, experts. Should AI replace software developers?

**Economist:** I believe AI can enhance productivity but not fully replace developers. 

**Tech Expert:** I agree; AI can automate tasks but lacks creativity.

**Economist:** However, AI could lead to job displacement.

**Tech Expert:** True, but it will create new roles too.

**Moderator:** Interesting points!

Create Output Schema

In [21]:
from pydantic import BaseModel, Field
from typing import List


class ExpertOpinion(BaseModel):
    role: str = Field(description="Expert role")

    opinion: str = Field(
        description="Expert opinion on the topic"
    )

    benefits: List[str] = Field(
        description="Benefits identified"
    )

    risks: List[str] = Field(
        description="Risks identified"
    )

    final_stance: str = Field(
        description="Final recommendation"
    )

In [22]:
from langchain_core.prompts import ChatPromptTemplate

expert_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a highly experienced {role}.

            Analyze the question carefully.

            Provide:
            1. Opinion
            2. Benefits
            3. Risks
            4. Final stance
            """
        ),
        (
            "human",
            """
            Question:

            {question}
            """
        )
    ]
)

In [23]:
structured_llm = llm.with_structured_output(
    ExpertOpinion
)

In [24]:
def generate_expert(role):
    
    chain = (
        expert_prompt
        | structured_llm
    )

    return chain.invoke(
        {
            "role": role,
            "question": question
        }
    )

In [25]:
from langchain_core.runnables import RunnableParallel

In [26]:
expert_chains = RunnableParallel(
    software_engineer=(
        expert_prompt
        | structured_llm
    ),
    ceo=(
        expert_prompt
        | structured_llm
    ),
    economist=(
        expert_prompt
        | structured_llm
    ),
    student=(
        expert_prompt
        | structured_llm
    ),
    ai_researcher=(
        expert_prompt
        | structured_llm
    ),
)

In [27]:
question = "Should AI replace software developers?"

opinions = expert_chains.invoke(
    {
        "question": question,
        "role": "placeholder"
    }
)

In [28]:
opinions

{'software_engineer': ExpertOpinion(role='AI Ethics Expert', opinion='AI should not fully replace software developers, but rather augment their capabilities. While AI can automate certain tasks, the creativity, problem-solving skills, and human intuition of developers are irreplaceable.', benefits=['Increased efficiency in coding tasks', 'Reduction in repetitive work', 'Ability to analyze large datasets quickly', 'Support for developers in debugging and testing'], risks=['Loss of jobs for entry-level developers', 'Over-reliance on AI leading to skill degradation', 'Potential for biased AI outputs', 'Security vulnerabilities if AI-generated code is not properly vetted'], final_stance='AI should be used as a tool to enhance the work of software developers, not as a replacement.'),
 'ceo': ExpertOpinion(role='AI Ethics Expert', opinion='AI should not fully replace software developers, but rather augment their capabilities. While AI can automate certain tasks, the creativity, problem-solvi

In [30]:
formatted_responses = []

for expert, opinion in opinions.items():
    formatted_responses.append({
        "expert": expert,
        "role": opinion.role,
        "opinion": opinion.opinion,
        "benefits": opinion.benefits,
        "risks": opinion.risks,
        "final_stance": opinion.final_stance
    })

print(formatted_responses)

[{'expert': 'software_engineer', 'role': 'AI Ethics Expert', 'opinion': 'AI should not fully replace software developers, but rather augment their capabilities. While AI can automate certain tasks, the creativity, problem-solving skills, and human intuition of developers are irreplaceable.', 'benefits': ['Increased efficiency in coding tasks', 'Reduction in repetitive work', 'Ability to analyze large datasets quickly', 'Support for developers in debugging and testing'], 'risks': ['Loss of jobs for entry-level developers', 'Over-reliance on AI leading to skill degradation', 'Potential for biased AI outputs', 'Security vulnerabilities if AI-generated code is not properly vetted'], 'final_stance': 'AI should be used as a tool to enhance the work of software developers, not as a replacement.'}, {'expert': 'ceo', 'role': 'AI Ethics Expert', 'opinion': 'AI should not fully replace software developers, but rather augment their capabilities. While AI can automate certain tasks, the creativity,

In [31]:
formatted_responses[0]

{'expert': 'software_engineer',
 'role': 'AI Ethics Expert',
 'opinion': 'AI should not fully replace software developers, but rather augment their capabilities. While AI can automate certain tasks, the creativity, problem-solving skills, and human intuition of developers are irreplaceable.',
 'benefits': ['Increased efficiency in coding tasks',
  'Reduction in repetitive work',
  'Ability to analyze large datasets quickly',
  'Support for developers in debugging and testing'],
 'risks': ['Loss of jobs for entry-level developers',
  'Over-reliance on AI leading to skill degradation',
  'Potential for biased AI outputs',
  'Security vulnerabilities if AI-generated code is not properly vetted'],
 'final_stance': 'AI should be used as a tool to enhance the work of software developers, not as a replacement.'}

In [32]:
import pandas as pd

df = pd.DataFrame(formatted_responses)
df

,expert,role,opinion,benefits,risks,final_stance
0,software_engineer,AI Ethics Expert,AI should not fully replace software developer...,"[Increased efficiency in coding tasks, Reducti...","[Loss of jobs for entry-level developers, Over...",AI should be used as a tool to enhance the wor...
1,ceo,AI Ethics Expert,AI should not fully replace software developer...,[Increased efficiency in coding and debugging ...,[Over-reliance on AI could lead to a decline i...,AI should be viewed as a tool to enhance the w...
2,economist,AI Ethics Expert,AI should not fully replace software developer...,[Increased efficiency in coding and debugging ...,[Over-reliance on AI could lead to a decline i...,AI should be viewed as a tool to enhance the w...
3,student,AI Ethics Expert,AI should not fully replace software developer...,[Increased efficiency in coding and debugging ...,[Over-reliance on AI could lead to a decline i...,AI should be viewed as a tool to enhance the w...
4,ai_researcher,AI Ethics Expert,AI should not fully replace software developer...,"[Increased efficiency in coding tasks, Reducti...","[Loss of jobs for developers, Over-reliance on...",AI should be used as a tool to assist software...


In [33]:
for expert, opinion in opinions.items():
    print(
        f"{expert}: {opinion.final_stance}\n"
        f"Benefits: {', '.join(opinion.benefits)}\n"
        f"Risks: {', '.join(opinion.risks)}\n"
    )

software_engineer: AI should be used as a tool to enhance the work of software developers, not as a replacement.
Benefits: Increased efficiency in coding tasks, Reduction in repetitive work, Ability to analyze large datasets quickly, Support for developers in debugging and testing
Risks: Loss of jobs for entry-level developers, Over-reliance on AI leading to skill degradation, Potential for biased AI outputs, Security vulnerabilities if AI-generated code is not properly vetted

ceo: AI should be viewed as a tool to enhance the work of software developers, not as a replacement.
Benefits: Increased efficiency in coding and debugging tasks, Ability to handle repetitive tasks, allowing developers to focus on more complex problems, Potential for faster development cycles and reduced time to market
Risks: Over-reliance on AI could lead to a decline in human skills and knowledge, AI may not fully understand the nuances of user requirements and context, Job displacement for entry-level positio